# Biohub random hide: 1500 iterations × three seeds, then full GEFF

Attach the Biohub competition train data; enable GPU and Internet. This notebook trains **three independent 1500-iteration runs** for sample `6bba_05b6850b`, with random hide seeds **42, 43, 44** and 20% hidden annotations. Each run creates its own `visible.geff`, checkpoint, ROI candidates, visible-only ILP calibration, and hidden-GT evaluation. The final table compares the three seeds at **4 µm**. Hidden metrics are exploratory because all runs train and evaluate on different withheld annotations from the same sequence.

The final export uses **seed 42, chosen in advance**, to infer all **100 frames × 64 × 256 × 256 voxels** and write `predicted_full.geff`. This is a Zarr directory; download its ZIP and extract it before opening with `napari-geff`. It is a prediction from a model trained on this same sample's visible GT, not a Kaggle submission or independent test. Full-volume inference can take substantially longer than the 20-frame ROI; progress is printed every five frames. All outputs are under `/kaggle/working/biohub_random_hide_multiseed/`.


In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import display

DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
ROOT = Path('/kaggle/working/biohub_random_hide_multiseed')
REPO = Path('/kaggle/working/linajea')
SAMPLE = '6bba_05b6850b'
SEEDS = (42, 43, 44)
EXPORT_SEED = 42                 # Fixed before inspecting hidden metrics
HIDE_RATE = 0.20
ITERATIONS = 1500
BRANCH = 'biohub-io'
PINNED_COMMIT = None            # Set to your pushed commit SHA for an exact code revision

assert EXPORT_SEED in SEEDS
for suffix in ('.zarr', '.geff'):
    if not (DATA_DIR / f'{SAMPLE}{suffix}').is_dir():
        raise FileNotFoundError(DATA_DIR / f'{SAMPLE}{suffix}')
ROOT.mkdir(parents=True, exist_ok=True)
print('Input:', DATA_DIR, 'Output:', ROOT, 'Seeds:', SEEDS)


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before running this notebook')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/hathanhv/linajea.git', str(REPO)], check=True)
COMMIT = PINNED_COMMIT or subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', COMMIT], check=True)
COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
(ROOT / 'commit.txt').write_text(COMMIT + '\n')
print('Code commit:', COMMIT, 'GPU:', torch.cuda.get_device_name(0))

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *map(str, args)], check=True)

pip('-r', REPO / 'requirements-biohub-io.txt')
pip('-r', REPO / 'requirements-biohub-train.txt')
pip('--no-deps', 'git+https://github.com/abred/daisy@ab7b82054126a50ab4ad5dcf2ef885b02d33aab8')
pip('--no-deps', 'git+https://github.com/Kainmueller-Lab/funlib.learn.torch@36ef666d16bddda0bf3146d4c0efbe9b95c8463d')
pip('-e', REPO, '--no-deps')
for required in ('linajea/biohub_full_export.py', 'run_scripts/export_biohub_full_geff.py'):
    if not (REPO / required).is_file():
        raise FileNotFoundError(f'Push updated biohub-io branch first: missing {required}')


## Split, train, reconstruct, and evaluate each seed

Each seed gets a separate output directory. The train command reads only that seed's `visible.geff`. The original GEFF is read by splitting and held-out evaluation; ILP calibration uses only visible GT. Rerunning a completed seed skips its checkpoint but recomputes its ROI metrics. Logs are saved beside each run.


In [ ]:
def run_logged(command, log_path, *, show_every=100):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            log.write(line); log.flush()
            if 'Batch: iteration=' in line:
                iteration = int(line.split('Batch: iteration=')[1].split(',')[0])
                if iteration == 1 or iteration % show_every == 0:
                    print(line.rstrip())
            elif ('Full inference:' in line or 'Traceback' in line or
                  'checkpoint' in line.lower() or 'Error' in line):
                print(line.rstrip())
        if process.wait() != 0:
            raise RuntimeError(f'Command failed; inspect {log_path}')

for seed in SEEDS:
    run_root = ROOT / f'seed_{seed}'
    run_root.mkdir(parents=True, exist_ok=True)
    print(f'\n===== SEED {seed}: split =====', flush=True)
    subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide', 'split',
                    '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
                    '--output-dir', str(run_root), '--hide-rate', str(HIDE_RATE),
                    '--seed', str(seed)], cwd=REPO, check=True)
    split = json.loads((run_root / 'split.json').read_text())
    print('ROI:', split['roi_start'], split['roi_shape'],
          'actual hide rate:', split['actual_hide_rate'])

    train_dir = run_root / 'train'
    checkpoint = train_dir / f'train_net_checkpoint_{ITERATIONS}'
    if not checkpoint.is_file():
        print(f'===== SEED {seed}: train {ITERATIONS} iterations =====', flush=True)
        run_logged([sys.executable, '-m', 'run_scripts.train_biohub_smoke',
                    '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
                    '--tracksfile', str(run_root / 'visible.geff'),
                    '--output-dir', str(train_dir), '--iterations', str(ITERATIONS),
                    '--checkpoint-stride', '500', '--seed', str(seed)],
                   train_dir / 'train.log')
    print('Checkpoint:', checkpoint)

    print(f'===== SEED {seed}: ROI inference =====', flush=True)
    run_logged([sys.executable, '-m', 'run_scripts.biohub_random_hide',
                'reconstruct', '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
                '--output-dir', str(run_root), '--iterations', str(ITERATIONS),
                '--device', 'cuda'], run_root / 'reconstruct.log')
    print(f'===== SEED {seed}: visible-only ILP calibration =====', flush=True)
    run_logged([sys.executable, '-m', 'run_scripts.biohub_random_hide',
                'calibrate', '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
                '--output-dir', str(run_root)], run_root / 'calibrate.log')
    for variant in ('original', 'calibrated'):
        subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide',
                        'evaluate', '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
                        '--output-dir', str(run_root), '--variant', variant],
                       cwd=REPO, check=True, stdout=(run_root / f'evaluate_{variant}.log').open('w'))
    print(f'Seed {seed} evaluation saved.', flush=True)


In [ ]:
rows = []
for seed in SEEDS:
    run_root = ROOT / f'seed_{seed}'
    split = json.loads((run_root / 'split.json').read_text())
    cand = json.loads((run_root / 'evaluation' / 'metrics.json').read_text())
    selected = json.loads((run_root / 'evaluation_calibrated' / 'metrics.json').read_text())
    c = cand['thresholds_um']['4.0']
    s = selected['thresholds_um']['4.0']
    cal = json.loads((run_root / 'prediction_calibrated' / 'calibration.json').read_text())
    rows.append({
        'seed': seed, 'actual_hide_rate': split['actual_hide_rate'],
        'candidate_nodes': cand['predicted_nodes'],
        'hidden_cell_candidate': f"{c['hidden_node_recovered']}/{c['hidden_node_count']}",
        'selected_nodes': selected['predicted_nodes'],
        'visible_cell_candidate': f"{cal['candidate_supported_visible_nodes']}/{cal['visible_nodes_in_roi']}",
        'visible_cell_selected': f"{cal['chosen']['visible_node_recovered']}/{cal['visible_nodes_in_roi']}",
        'visible_edge_candidate': f"{cal['candidate_supported_visible_edges']}/{cal['visible_edges_in_roi']}",
        'visible_edge_selected': f"{cal['chosen']['visible_edge_recovered']}/{cal['visible_edges_in_roi']}",
        'hidden_cell_selected': f"{s['hidden_node_recovered']}/{s['hidden_node_count']}",
        'hidden_edge_candidate': f"{c['hidden_edge_recovered']}/{c['hidden_edge_count']}",
        'hidden_edge_selected': f"{s['hidden_edge_recovered']}/{s['hidden_edge_count']}",
        'hidden_confirmed_rate_lower_bound': s['hidden_confirmation_rate_lower_bound'],
        'ILP_selection_constant': cal['chosen']['selection_constant'],
        'ILP_track_cost': cal['chosen']['track_cost'],
    })
table = pd.DataFrame(rows).set_index('seed')
display(table)
table.to_csv(ROOT / 'summary_4um.csv')
(ROOT / 'summary.json').write_text(json.dumps(rows, indent=2))
print('These exploratory results use different hidden subsets for each seed.')
print('Output:', ROOT / 'summary_4um.csv')


## Export a full 100-frame prediction

This stage runs only the preselected seed 42 model. It streams output maps frame by frame, caches the candidate CSV, builds temporal edges, and solves independent candidate graph components using the visible-only calibrated ILP weights. The output `.geff` covers the full sample, not just the evaluation ROI. The directory is also zipped for downloading from Kaggle. If the cell is interrupted after the candidate CSV was saved, rerunning it reuses those candidates.


In [ ]:
import shutil
run_root = ROOT / f'seed_{EXPORT_SEED}'
export_dir = run_root / 'full_export'
geff = export_dir / 'predicted_full.geff'
if not geff.is_dir():
    run_logged([sys.executable, '-m', 'run_scripts.export_biohub_full_geff',
                '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
                '--run-dir', str(run_root), '--iterations', str(ITERATIONS),
                '--device', 'cuda'], export_dir / 'run.log')
report = json.loads((export_dir / 'export.json').read_text())
zip_path = Path(shutil.make_archive(str(export_dir / 'predicted_full_geff'), 'zip',
                                    root_dir=export_dir, base_dir=geff.name))
print('Predicted graph:', report)
print('GEFF directory:', geff)
print('Downloadable ZIP:', zip_path, 'bytes:', zip_path.stat().st_size)


In [ ]:
import zarr
from linajea.biohub_io import BiohubImage, read_geff_tracks
image = BiohubImage(DATA_DIR / f'{SAMPLE}.zarr')
records, edge_count = read_geff_tracks(geff, image.voxel_size, image.shape)
print('GEFF round-trip:', len(records), 'nodes,', edge_count, 'edges')
print('Image shape / voxel size:', image.shape, image.voxel_size)
print('First predicted records:', records[:3])
print('Open the extracted predicted_full.geff folder in napari with napari-geff.')
print('Code revision:', COMMIT)
